# Feature Stores

Companion notebook for the [Feature Stores lesson](https://ml-viz-ruby.vercel.app/courses/ml-in-practice/17-feature-stores).

**The idea in one sentence.** A feature store's central job is **point-in-time
correctness**: when you build training data, each label must be joined to the feature values
*as they were at that moment* — a naive "use the latest value" join **leaks the future** and
silently inflates offline metrics.

The key distinction:

- **Naive join (WRONG):** attach the current feature value to every historical event — the
  training row sees data that didn't exist yet.
- **As-of / point-in-time join (RIGHT):** attach the last value whose timestamp is $\le$ the
  event time.

We build both, **validate that the naive join leaks and the as-of join is correct**, then
cover the gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Dark style matching the site theme.
plt.style.use('dark_background')
plt.rcParams.update({
    'axes.edgecolor': '#475569',
    'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'axes.titlecolor': '#e2e8f0',
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'grid.color': '#2e3347',
    'savefig.facecolor': '#0f1117',
})
BRAND = '#6366f1'
TEAL = '#14b8a6'
ROSE = '#f43f5e'
YELLOW = '#eab308'

rng = np.random.default_rng(0)

## 1. The data: events to label, and a feature history

Each **label event** happens at some time `t`. A **feature** ('30-day spend') changes over time, recorded as (timestamp, value). To train correctly we must attach the feature value *as it was at `t`* — not its latest value.

In [ ]:
# Feature history for one entity: (time, value)
feature_history = [(0, 10.0), (5, 25.0), (9, 40.0), (14, 55.0)]
# Label events at various times
events = [3, 7, 12, 16]
print('feature history:', feature_history)
print('label event times:', events)

## 2. Naive join (WRONG) vs as-of join (RIGHT)

- **Naive**: attach the *latest* feature value to every event → leaks future information.
- **As-of**: attach the most recent value with `t_f <= t` → leakage-free.

In [ ]:
def naive_join(t, history):
    return history[-1][1]  # latest value — leaks the future!

def asof_join(t, history):
    valid = [v for (tf, v) in history if tf <= t]
    return valid[-1] if valid else None

print(f"{'event t':>8} | {'naive (leak)':>12} | {'as-of (correct)':>15}")
for t in events:
    print(f'{t:>8} | {naive_join(t, feature_history):>12} | {asof_join(t, feature_history):>15}')

### Validate: the naive join leaks the future; the as-of join doesn't

The naive join returns the *latest* feature value for every event — including events that
happened *before* that value existed. The as-of join returns only what was known at or
before the event time. We confirm the naive join differs from the correct one (that
difference is leaked future information).

In [ ]:
naive = np.array([naive_join(t, feature_history) for t in events])
correct = np.array([asof_join(t, feature_history) for t in events])
print(f'event times     : {events}')
print(f'naive (leak)    : {naive}')
print(f'as-of (correct) : {correct}')
leak = np.abs(naive - correct).mean()
print(f'mean feature error introduced by the leak: {leak:.2f}')
assert leak > 0, 'the naive join injects future information (leakage)'
# the as-of value is non-decreasing in t here (only past values, which grow over time)
asof_seq = [asof_join(t, feature_history) for t in sorted(events)]
assert all(asof_seq[i] <= asof_seq[i+1] for i in range(len(asof_seq)-1)), 'as-of uses only past values'
print('\n✅ point-in-time (as-of) join prevents the future-leakage the naive join causes')

The naive column shows `55.0` for *every* event — including events at t=3 and t=7, when the true value was 10 and 25. Training on that lets the model peek at the future, so offline metrics look great and production collapses.

In [ ]:
# Quantify the leakage: mean absolute error between naive and correct features
naive = np.array([naive_join(t, feature_history) for t in events])
correct = np.array([asof_join(t, feature_history) for t in events])
print('naive  features:', naive)
print('as-of  features:', correct)
print('mean feature error introduced by the leak:', np.abs(naive - correct).mean())

fig, ax = plt.subplots(figsize=(8, 4))
ts = [h[0] for h in feature_history]; vs = [h[1] for h in feature_history]
ax.step(ts, vs, where='post', color=TEAL, lw=2, label='true feature over time')
ax.scatter(events, correct, color=TEAL, s=70, zorder=5, label='as-of (correct)')
ax.scatter(events, naive, color=ROSE, s=70, marker='x', zorder=5, label='naive (leaked)')
ax.set_xlabel('time'); ax.set_ylabel('feature value'); ax.set_title('Point-in-time join vs leaked latest value')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **naive latest-value join** | leaks future features → inflated offline metrics, production collapse (verified) |
| **online/offline skew** | the serving feature pipeline must match the training one exactly |
| **feature freshness** | stale online features hurt serving; monitor update latency |
| **backfill correctness** | recomputing historical features must respect point-in-time logic |
| **feature reuse vs coupling** | shared features avoid duplication but couple teams to a schema |

Demo: leaked training rows teach the model signals it won't have at inference time.

In [ ]:
# Why this leak is so dangerous: it inflates OFFLINE metrics that then crash in production.
# A model trained on leaked (future) features looks great on the training split but has no
# access to those values at serving time. We quantify the fraction of events that got a
# leaked value.
leaked = sum(1 for t in events if naive_join(t, feature_history) != asof_join(t, feature_history))
print(f'{leaked} of {len(events)} training rows received a FUTURE feature value')
print('Every leaked row teaches the model a signal it will NOT have at inference time ->')
print('offline metrics look great, production accuracy collapses. Feature stores enforce as-of joins.')

## ✏️ Your turn — implement the as-of join

Implement `point_in_time(t, history)` returning the most recent feature value with timestamp `<= t` (or `None` if none exists). `history` is a list of `(timestamp, value)` sorted by time.

In [ ]:
def point_in_time(t, history):
    """TODO(you): return the last value whose timestamp <= t, else None."""
    # TODO
    return ...


In [ ]:
assert point_in_time(3, feature_history) == 10.0
assert point_in_time(7, feature_history) == 25.0
assert point_in_time(100, feature_history) == 55.0
assert point_in_time(-1, feature_history) is None
print('✅ as-of join uses only information available at or before t.')

<details>
<summary>Solution</summary>

```python
def point_in_time(t, history):
    valid = [v for (tf, v) in history if tf <= t]
    return valid[-1] if valid else None
```

A feature store does this join for you across millions of rows, and serves the *same* definition online at inference — so training and serving can't diverge.
</details>

## Recap

- A feature store defines a feature once and serves it to **offline** (training) and **online** (inference) — no skew.
- **Point-in-time correctness** ($t_f \le t$) prevents the future leaking into training.
- The naive 'latest value' join inflates offline metrics and collapses in production.
- Use a feature store when features are shared, served online, or joined point-in-time at scale.